In [2]:
import numpy as np
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec
from tabulate import tabulate
from collections import Counter

# Download stopwords silently
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

# Define the initial dataset [cite: 24-28]
dataset = [
    "I love playing football on the weekends",
    "I enjoy hiking and camping in the mountains",
    "I like to read books and watch movies",
    "I prefer playing video games over sports",
    "I love listening to music and going to concerts"
]

k = 2 # Set the number of clusters
print("Cell 1 complete! Libraries loaded and dataset is ready.")

Cell 1 complete! Libraries loaded and dataset is ready.


In [3]:
# 1. Vectorize using TF-IDF [cite: 29-31]
vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(dataset)

# 2. Cluster using KMeans [cite: 32-37]
km_tfidf = KMeans(n_clusters=k, random_state=42)
y_pred_tfidf = km_tfidf.fit_predict(X_tfidf)

# 3. Display Results [cite: 38-40]
print("--- TF-IDF CLUSTERING ---")
table_tfidf = [["Document", "Predicted Cluster"]]
table_tfidf.extend([[doc, cluster] for doc, cluster in zip(dataset, y_pred_tfidf)])
print(tabulate(table_tfidf, headers="firstrow"))

# 4. Calculate Purity [cite: 53-57]
purity_tfidf = sum(max(cluster.values()) for cluster in [Counter(y_pred_tfidf)]) / len(y_pred_tfidf)
print(f"\nTF-IDF Purity: {purity_tfidf}")

--- TF-IDF CLUSTERING ---
Document                                           Predicted Cluster
-----------------------------------------------  -------------------
I love playing football on the weekends                            0
I enjoy hiking and camping in the mountains                        0
I like to read books and watch movies                              1
I prefer playing video games over sports                           0
I love listening to music and going to concerts                    1

TF-IDF Purity: 0.6


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


In [4]:
# 1. Train Word2Vec Model [cite: 102-104]
tokenized_dataset = [doc.split() for doc in dataset]
word2vec_model = Word2Vec(sentences=tokenized_dataset, vector_size=100, window=5, min_count=1, workers=4)

# 2. Create Embeddings [cite: 105-106]
X_w2v = np.array([np.mean([word2vec_model.wv[word] for word in doc.split() if word in word2vec_model.wv], axis=0) for doc in dataset])

# 3. Cluster [cite: 107-112]
km_w2v = KMeans(n_clusters=k, random_state=42)
y_pred_w2v = km_w2v.fit_predict(X_w2v)

# 4. Display Results [cite: 113-115]
print("--- WORD2VEC CLUSTERING ---")
table_w2v = [["Document", "Predicted Cluster"]]
table_w2v.extend([[doc, cluster] for doc, cluster in zip(dataset, y_pred_w2v)])
print(tabulate(table_w2v, headers="firstrow"))

# 5. Calculate Purity [cite: 119-124]
purity_w2v = sum(max(cluster.values()) for cluster in [Counter(y_pred_w2v)]) / len(y_pred_w2v)
print(f"\nWord2Vec Purity: {purity_w2v}")

--- WORD2VEC CLUSTERING ---
Document                                           Predicted Cluster
-----------------------------------------------  -------------------
I love playing football on the weekends                            1
I enjoy hiking and camping in the mountains                        0
I like to read books and watch movies                              1
I prefer playing video games over sports                           0
I love listening to music and going to concerts                    1

Word2Vec Purity: 0.6


In [5]:
# Create the preprocessing function
def preprocess(text):
    if not isinstance(text, str): # Safety check for empty rows
        return ""
    text = text.lower() # Lowercase
    text = re.sub(r'[^\w\s]', '', text) # Remove punctuation
    text = " ".join([word for word in text.split() if word not in stop_words]) # Remove stopwords
    return text

# Apply cleaning to the dataset
cleaned_dataset = [preprocess(doc) for doc in dataset]

# Re-run TF-IDF on the cleaned data
X_clean_tfidf = vectorizer.fit_transform(cleaned_dataset)
km_clean_tfidf = KMeans(n_clusters=k, random_state=42)
y_pred_clean_tfidf = km_clean_tfidf.fit_predict(X_clean_tfidf)

# Calculate new purity
print("--- EXERCISE 1: PREPROCESSING ---")
purity_clean = sum(max(cluster.values()) for cluster in [Counter(y_pred_clean_tfidf)]) / len(y_pred_clean_tfidf)
print(f"Cleaned TF-IDF Purity: {purity_clean}")

--- EXERCISE 1: PREPROCESSING ---
Cleaned TF-IDF Purity: 0.8


In [8]:
# Load the dataset
df = pd.read_csv('customer_complaints_1.csv')

# 🛑 Only use the column that contains the actual complaints
column_name = 'text' 

# Pre-process the text column using the function we made in Cell 4
df['cleaned_text'] = df[column_name].apply(preprocess)

# Vectorize the complaints
vectorizer_csv = TfidfVectorizer()
X_csv = vectorizer_csv.fit_transform(df['cleaned_text'])

# Cluster into 3 groups
km_csv = KMeans(n_clusters=3, random_state=42) 
df['Cluster'] = km_csv.fit_predict(X_csv)

# Show the top 10 results
print("--- EXERCISE 2: CSV CLUSTERING ---")
print(df[[column_name, 'Cluster']].head(10))

--- EXERCISE 2: CSV CLUSTERING ---
                                                text  Cluster
0  I used to love Comcast. Until all these consta...        1
1  I'm so over Comcast! The worst internet provid...        2
2  If I could give them a negative star or no sta...        0
3  I've had the worst experiences so far since in...        0
4  Check your contract when you sign up for Comca...        2
5  Thank God. I am changing to Dish. They gave me...        1
6  I Have been a long time customer and only have...        1
7  There is a malfunction on the DVR manager whic...        0
8  Charges overwhelming. Comcast service rep was ...        1
9  I have had cable, DISH, and U-verse, etc. in t...        1
